In [1]:
#!pip install pyspark==3.5.5 tables snakebite-py3

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import h5py
import dbutils
import requests
import os
import pyspark
from pyspark.sql import functions as F
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
HDFS_HOST = "192.168.2.31"
HDFS_PORT = 9000
HDFS_BASE = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"
import os
import sys
os.environ['PYSPARK_PYTHON'] = "python3"
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
from pyspark.sql import SparkSession

spark_session :SparkSession = SparkSession.builder \
    .master("spark://192.168.2.31:7077") \
    .appName("spark_preprocess_driver") \
    .config("spark.dynamicAllocation.enabled", True) \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", True) \
    .config("spark.shuffle.service.enabled", False) \
    .config("spark.dynamicAllocation.executorIdleTimeout", "30s") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()
    
spark_context = spark_session.sparkContext
spark_context.setLogLevel("ERROR")

#Get this file by:
#1. running "mkdir snakebite", "cd snakebite", "pip install snakebite-py3 -t .", "zip -r ../snakebite.zip .", then putting its path as an argument below
spark_context.addPyFile("./spark_dependencies/snakebite.zip")
#the pip package tables needs to be installed for the file below to work
spark_context.addPyFile("hdf5_getters.py")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/18 10:22:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/18 10:22:29 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.


In [3]:
from snakebite.client import Client
HDFS_HOST = "192.168.2.31"
HDFS_PORT = 9000
HDFS_BASE = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"

client = Client(HDFS_HOST, HDFS_PORT)


/tmp/spark-e3fcc010-7068-4eb5-ad46-21005177f8c9/userFiles-b3949053-d2d8-47ea-aec4-b904a559d51b/snakebite.zip/snakebite/client.py:816: SyntaxWarning: "is not" with a literal. Did you mean "!="?
/tmp/spark-e3fcc010-7068-4eb5-ad46-21005177f8c9/userFiles-b3949053-d2d8-47ea-aec4-b904a559d51b/snakebite.zip/snakebite/client.py:816: SyntaxWarning: "is not" with a literal. Did you mean "!="?


In [4]:
starting_directory = "/data/MillionSongSubset"

done = False

#NOTE: in case of large amounts of files, this will likely make the driver run out of memory.
#In that case, recursion has to be implemented manually. 
all_files = list(client.ls(["/data/MillionSongSubset"], recurse=True))

In [5]:
h5_files = []
for file in all_files:
    if file["file_type"] != "f":
        continue
    
    if file["path"].split(".")[-1] == "h5":
        h5_files.append(file["path"])
        

In [6]:
from preprocessing.hdf5_getters import get_desired
import io
import tables
import tempfile

#transformation using pytables
def get_relevant_metadata_of_song_file(file_path):
    HDFS_HOST = "192.168.2.31"
    HDFS_PORT = 9000
    client = Client(HDFS_HOST, HDFS_PORT)
    binary_data = b''.join(list(client.cat([file_path]))[0])

    #Change this according to needs. Check for available fields at the bottom of the hdf5_getters file
    RELEVANT_FIELDS = [
    'artist_name',
    'title',
    'duration',
    'year',
    'song_id'
    ]
    
    file_contents = io.BytesIO(binary_data)
    

    with tempfile.NamedTemporaryFile(delete=True) as temp_file:
        temp_file.write(file_contents.getvalue())
        temp_file_path = temp_file.name
        
        file = tables.open_file(temp_file_path)
        song_metadata = get_desired(file, RELEVANT_FIELDS)

    relevant_data = {}
    
    for field in RELEVANT_FIELDS:
        relevant_data[field] = str(song_metadata[field])
    
    return relevant_data

#transformation using h5py
def get_song_name(binary_data):
    import h5py
    file = h5py.File(binary_data)
    song_title = file["metadata"]["songs"][0][18]
    return song_title

In [7]:
rdd = spark_context.parallelize(h5_files)

rdd.top(3)

['/data/MillionSongSubset/B/I/J/TRBIJYB128F14AE326.h5',
 '/data/MillionSongSubset/B/I/J/TRBIJRN128F425F3DD.h5',
 '/data/MillionSongSubset/B/I/J/TRBIJNK128F93093EC.h5']

In [8]:
relevant_data = rdd.map(get_relevant_metadata_of_song_file)
df = spark_session.createDataFrame(relevant_data).persist()

In [ ]:
#FILENAME_OUTPUT = "song_metadata_preprocessing_2.csv"
#df.write.csv(f"{HDFS_BASE}/{FILENAME_OUTPUT}")


25/03/18 11:20:07 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
25/03/18 11:20:09 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce

In [ ]:
sqlContext = SQLContext(spark_session.sparkContext)

In [ ]:
user_data = sqlContext.read.csv("hdfs://192.168.2.31:9000/data/train_triplets.txt", sep="\t", header=False, inferSchema=True).cache()

In [ ]:
user_data = user_data.toDF("user_id", "song_id", "play_count")
user_data.show(10)
most_active_user = (
    user_data.groupBy("user_id")
    .agg(F.sum("play_count").alias("Total_plays"))
    .orderBy(F.desc("Total_plays"))
    .limit(1)
)
most_active_user.collect()[0][0]

In [ ]:
songs_data = songs_data.toDF("artist","duration", "song_id", "song title", "release")
songs_data.show(10)
songs_data.printSchema()


In [ ]:
from pyspark.sql import functions as F

def remove_byte_prefix(songs_df):
    for column in songs_df.columns:
        songs_df = songs_df.withColumn(
            column, F.regexp_replace(F.col(column), r"^b[\"']|[\"']$", "")
        )
    return songs_df


user_data = remove_byte_prefix(user_data)
songs_data = remove_byte_prefix(songs_data)

songs_data.show(10)

In [ ]:
merged_df = songs_data.join(user_data, on='song_id', how='inner')
merged_df.show(10)

In [ ]:
from pyspark.sql import functions as F
def top_songs(merged_data, user, num_songs = 3):
    user_data = merged_data.filter(merged_data['user_id'] == user)
    top_user_songs = user_data.groupBy('song title').agg(F.sum('play_count').alias('Total_plays')).orderBy(F.desc('Total_plays')).limit(num_songs)
    return top_user_songs

def top_artists(merged_data, user, num_artists = 3):
    user_data = merged_data.filter(merged_data['user_id'] == user)
    top_user_artists = user_data.groupBy('artist').agg(F.countDistinct('song_id').alias('number_of_songs')).orderBy(F.desc('number_of_songs')).limit(num_artists)
    return top_user_artists

In [ ]:
# Example Usage
top_song = top_songs(merged_df, user = "093cb74eb3c517c5179ae24caf0ebec51b24d2a2")
top_song.show()
top_artist = top_artists(merged_df, user = "093cb74eb3c517c5179ae24caf0ebec51b24d2a2")
top_artist.show()

In [ ]:
spark_session.stop()